In [132]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score,cross_val_predict
from sklearn.metrics import accuracy_score, f1_score,recall_score,precision_score, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

In [133]:
#importinng the appointment_clean.csv file
data = pd.read_csv('appointments_clean.csv')
data.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


# Task 3.1 Features, split and baseline

In [134]:
# Feature Selection
X = data[['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent','booking_channel','visit_type']]

print('Number of rows and columns selected as basic rows and columns',X.shape)
X.head()

Number of rows and columns selected as basic rows and columns (990, 8)


,age,distance_km,fee_inr,lead_days,prior_no_shows,reminder_sent,booking_channel,visit_type
0,60,18.0,850,1,1,Yes,Phone,Consultation
1,39,12.5,350,10,2,No,Walk-In,Follow-up
2,63,4.1,850,6,0,No,Walk-In,Consultation
3,52,7.5,750,4,0,Yes,App,Consultation
4,64,19.0,700,40,0,No,Phone,Consultation


In [135]:
# Feature Engineering
X['reminder_sent'] = X['reminder_sent'].map({'Yes':1,'No':0})

X= pd.get_dummies(X, columns=["booking_channel","visit_type"]).astype(int)

print('Number of rows and columns after feature engineering',X.shape)
X.head()

Number of rows and columns after feature engineering (990, 11)


,age,distance_km,fee_inr,lead_days,prior_no_shows,reminder_sent,booking_channel_App,booking_channel_Phone,booking_channel_Walk-In,visit_type_Consultation,visit_type_Follow-up
0,60,18,850,1,1,1,0,1,0,1,0
1,39,12,350,10,2,0,0,0,1,0,1
2,63,4,850,6,0,0,0,0,1,1,0
3,52,7,750,4,0,1,1,0,0,1,0
4,64,19,700,40,0,0,0,1,0,1,0


In [136]:
# Target variable selection
y = data['no_show']

In [137]:
# Spliting the data into Train and Test data
x_train, x_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

print(f'Training Size- X: {x_train.shape[0]}, Y:{y_train.shape[0]}')
print(f'Test Size- X: {x_test.shape[0]}, Y:{y_test.shape[0]}')
print('')
print('Number of no_show (1) in test set: ',y_test.value_counts()[1])

Training Size- X: 792, Y:792
Test Size- X: 198, Y:198

Number of no_show (1) in test set:  49


In [138]:
# total no show in the test set
print('Number of no_show (1) in test set: ',y_test.value_counts()[1])

Number of no_show (1) in test set:  49


In [139]:
# Base prediction where model predicted Attended every time
base_pred = [0]*198

#Accuracy score
accuracy_score(y_test, base_pred)

0.7525252525252525

# Task 3.2 Logistic Regression

In [140]:
# Normalising the data
scaler = StandardScaler()
x_train_s = scaler.fit_transform(x_train)               # fit and transform on training data
x_test_s = scaler.transform(x_test)                     # only transform on test data

In [141]:
# Creating the Logistic Regression model
logistic_r = LogisticRegression(max_iter=1000)

logistic_r.fit(x_train_s,y_train)                       # fitting it on training data
pred_lr = logistic_r.predict(x_test_s)                  # predicting the result of test data

y_probs_lr = logistic_r.predict_proba(x_test_s)[:, 1]   # the probability we got from the model on Test data 

con = confusion_matrix(y_test,pred_lr)                  # Creating Confusion  Matrics
print('Confusion matrics: \n',con)

Confusion matrics: 
 [[139  10]
 [ 32  17]]


In [142]:
# Printing the Validation Score achieved by Logistic Regression model
print(f'''
This Model achieves the following Score -
        Accuracy score:  {accuracy_score(y_test,pred_lr):.2f}
        Precision score: {precision_score(y_test,pred_lr):.2f}
        Recall score:    {recall_score(y_test,pred_lr):.2f}
        F1 score:        {f1_score(y_test,pred_lr):.2f}
        ROC-AUC score:   {roc_auc_score(y_test,y_probs_lr):.2f}
'''
)


This Model achieves the following Score -
        Accuracy score:  0.79
        Precision score: 0.63
        Recall score:    0.35
        F1 score:        0.45
        ROC-AUC score:   0.79



In [143]:
# Checking Coefficients for future use 
coefficient = logistic_r.coef_[0]
coefficient

array([-0.25508759,  0.26507045,  0.45234791,  0.61528546,  0.65088383,
       -0.58171473, -0.0468978 ,  0.21603724, -0.18643122,  0.0157196 ,
       -0.0157196 ])

In [144]:
# Creating a DataFrame from the features name, coefficient and the exponential 
print('the coefficient includes:')
for i in range(len(X.columns)):
    coef_df = pd.DataFrame({
        'feature': X.columns,
        'coefficient': coefficient.round(2),
        'exponential': np.exp(coefficient).round(2)
})

print(coef_df)

the coefficient includes:
                    feature  coefficient  exponential
0                       age        -0.26         0.77
1               distance_km         0.27         1.30
2                   fee_inr         0.45         1.57
3                 lead_days         0.62         1.85
4            prior_no_shows         0.65         1.92
5             reminder_sent        -0.58         0.56
6       booking_channel_App        -0.05         0.95
7     booking_channel_Phone         0.22         1.24
8   booking_channel_Walk-In        -0.19         0.83
9   visit_type_Consultation         0.02         1.02
10     visit_type_Follow-up        -0.02         0.98


In this model, if we consider `remainder_sent` column for an example, and hold the other features as constant. By calculating the exponential of the coefficient (i.e. __-58__) of `remainder_sent` column, we get __0.56__ as result. It means that every 1 scaler value increase in `remainder_sent` column multiplies the odds of no-show by 0.56.

But this is not the full correct answer for reaching the correct answer we have to multiply the coefficient with the **Scale of Standard Scaler**. And divide the coefficient with the value of standard scaler to get **adjusted coefficient**. using this **adjusted coefficient** for calculating exponential gives us the true multiplier.

In [145]:
# Adding Values of Standard Scaler to the Coefficient DataFrame.
for feature, scale in zip(X.columns, scaler.scale_):
    print(feature, ":", scale.round(2))

coef_df['scaler'] = scaler.scale_.round(2)

age : 16.66
distance_km : 6.03
fee_inr : 184.25
lead_days : 8.61
prior_no_shows : 0.97
reminder_sent : 0.49
booking_channel_App : 0.49
booking_channel_Phone : 0.48
booking_channel_Walk-In : 0.44
visit_type_Consultation : 0.49
visit_type_Follow-up : 0.49


In [146]:
# Calculating the adjusted coefficient by dividing values of coefficient column with the values of scaler column.
coef_df['adjusted_coef'] = round(coef_df['coefficient'] / coef_df['scaler'],2)

coef_df

,feature,coefficient,exponential,scaler,adjusted_coef
0,age,-0.26,0.77,16.66,-0.02
1,distance_km,0.27,1.30,6.03,0.04
2,fee_inr,0.45,1.57,184.25,0.00
3,lead_days,0.62,1.85,8.61,0.07
4,prior_no_shows,0.65,1.92,0.97,0.67
5,reminder_sent,-0.58,0.56,0.49,-1.18
6,booking_channel_App,-0.05,0.95,0.49,-0.10
7,booking_channel_Phone,0.22,1.24,0.48,0.46
8,booking_channel_Walk-In,-0.19,0.83,0.44,-0.43
9,visit_type_Consultation,0.02,1.02,0.49,0.04


In [147]:
# Now caalculating the Real adjusted exponential
coef_df['adjusted_expo'] = np.exp(coef_df['adjusted_coef']).round(2)

coef_df

,feature,coefficient,exponential,scaler,adjusted_coef,adjusted_expo
0,age,-0.26,0.77,16.66,-0.02,0.98
1,distance_km,0.27,1.30,6.03,0.04,1.04
2,fee_inr,0.45,1.57,184.25,0.00,1.00
3,lead_days,0.62,1.85,8.61,0.07,1.07
4,prior_no_shows,0.65,1.92,0.97,0.67,1.95
5,reminder_sent,-0.58,0.56,0.49,-1.18,0.31
6,booking_channel_App,-0.05,0.95,0.49,-0.10,0.90
7,booking_channel_Phone,0.22,1.24,0.48,0.46,1.58
8,booking_channel_Walk-In,-0.19,0.83,0.44,-0.43,0.65
9,visit_type_Consultation,0.02,1.02,0.49,0.04,1.04


Now by reconsider `remainder_sent` column. By calculating adjusted coefficient in `adjusted_coef` column, by dividing the coefficient (i.e. __-58__) to the scaler value (i.e. __0.49__) produces adjusted coefficient (i.e.__-1.18__). Using this we calculate the adjusted exponential in `adjusted_expo` column with value being __0.31__ as result. It means that:

**Every 1 scaler value increase in `remainder_sent` column multiplies the odds of no-show by 0.31.**


### Why do they differ?
The first odds ratio is based on a one-standard-deviation increase in the scaled variable, while the second odds ratio represents the effect of changing reminder_sent from 0 to 1 in the original data, so the values differ because the model was trained on standardized features.

### Final Result
**In this model the odds of a no-show are about 69% lower for appointments with a reminder,when holding the other features constant — __an association, not proof of cause.__**

# Task 3.3 Decision Tree

In [148]:
# Creating a Decision Tree Classifier model with max depth = None
dec_tree = DecisionTreeClassifier(random_state=42)

dec_tree.fit(x_train_s,y_train)
pred_dt_train = dec_tree.predict(x_train_s)
pred_dt_test = dec_tree.predict(x_test_s)

In [149]:
# printing the results 
print('Checking the Result for Training Data when max depth = None\n')
print(f'''                                                                                                                      
This Model achieves the following Score -
        Accuracy score:  {accuracy_score(y_train,pred_dt_train):.2f}
'''
)                                                                       # Score of Decision Tree model with max depth = None on training data 
print('-'*50)
print('Checking the Result for Test Data when max depth = None\n')
print(f'''
This Model achieves the following Score -
        Accuracy score:  {accuracy_score(y_test,pred_dt_test):.2f}
'''
)                                                                       # Score of Decision Tree Model with max depth = None on Test data

Checking the Result for Training Data when max depth = None

                                                                                                                      
This Model achieves the following Score -
        Accuracy score:  1.00

--------------------------------------------------
Checking the Result for Test Data when max depth = None


This Model achieves the following Score -
        Accuracy score:  0.67



In [150]:
# Creating a Decision Tree Classifier madel with max depth =3
dec_tree = DecisionTreeClassifier(max_depth=3,random_state=42)

dec_tree.fit(x_train_s,y_train)
pred_dt_train = dec_tree.predict(x_train_s)
pred_dt_test = dec_tree.predict(x_test_s)
y_probs_dt = dec_tree.predict_proba(x_test_s)[:, 1]                 # Calculating probability for calculating ROC-AUC score

In [151]:
print('Checking the Result for Training Data when max depth is 3\n')
print(f'''
This Model achieves the following Score -
        Accuracy score:  {accuracy_score(y_train,pred_dt_train):.2f}
'''
)                                                                   # Score of Decision Tree Model with max depth = 3 on Test data
print('-'*50)
print('Checking the Result for Test Data when max depth is 3\n')
print(f'''
This Model achieves the following Score -
        Accuracy score:  {accuracy_score(y_test,pred_dt_test):.2f}
'''
)                                                                   # Score of Decision Tree Model with max depth = 3 on Test data

Checking the Result for Training Data when max depth is 3


This Model achieves the following Score -
        Accuracy score:  0.82

--------------------------------------------------
Checking the Result for Test Data when max depth is 3


This Model achieves the following Score -
        Accuracy score:  0.78



The large gap between the training accuracy (1.00) and test accuracy (0.67) for the *unrestricted tree* shows the sign of `overfitting` by the model, while the *depth-3 tree* `generalizes` better because its training and test accuracies are much closer (0.82 vs 0.78).

# Task 3.4 Random forest and a comparison table

In [152]:
# Creating a Random Forest Classifier model
random_f = RandomForestClassifier(n_estimators=100,random_state=42)
random_f.fit(x_train_s,y_train)

pred_rfc_train = random_f.predict(x_train_s)                        # Prediction on Training data 
pred_rfc_test = random_f.predict(x_test_s)                          # Prediction on Test data

y_probs_rfc = random_f.predict_proba(x_test_s)[:, 1]                # Calculating probability for calculating ROC-AUC score

In [153]:
print('Checking the Result for Training Data \n')
print(f'''
This Model achieves the following Score -
        Accuracy score:  {accuracy_score(y_train,pred_rfc_train):.2f}
        Recall score:    {recall_score(y_train,pred_rfc_train):.2f}
'''
)                                                                               # Printing scores for the model, tested on train data

print('-'*50)
print('Checking the Result for Test Data \n')
print(f'''
This Model achieves the following Score -
        Accuracy score:  {accuracy_score(y_test,pred_rfc_test):.2f}
        Recall score:    {recall_score(y_test,pred_rfc_test):.2f}
        ROC-AUC score:   {roc_auc_score(y_test,y_probs_rfc):.2f}
'''
)                                                                               # Printing scores for the model, tested on test data

Checking the Result for Training Data 


This Model achieves the following Score -
        Accuracy score:  1.00
        Recall score:    1.00

--------------------------------------------------
Checking the Result for Test Data 


This Model achieves the following Score -
        Accuracy score:  0.78
        Recall score:    0.37
        ROC-AUC score:   0.71



In [154]:
# Calculating Feature Importance
imp_feature = random_f.feature_importances_
print(imp_feature)

[0.2165963  0.1657619  0.14192785 0.22671044 0.10511609 0.0456507
 0.01961508 0.02615356 0.01898213 0.01624556 0.01724038]


In [155]:
# Creating the DataFrame from Features and Feature Importance for printing Top 3 Important Features
for i in range(len(X.columns)):
    feature_df = pd.DataFrame({
        'feature': X.columns,
        'feature_imp': random_f.feature_importances_.round(2)
})

print(f'Top 3 important features: \n{feature_df.sort_values(axis=0, by='feature_imp' ,ascending=False)[0:3]}')      #Printing top 3 Features


Top 3 important features: 
       feature  feature_imp
3    lead_days         0.23
0          age         0.22
1  distance_km         0.17


In [156]:
# Creating a Table with the result(scores) of all previously made model, including: Logistic Regression, Decision Tree, and Random Forest 
score_table = {
    'Model': ['LogisticRegression','DecisionTree','RandomForest'],
    'accuracy_score': [
        round(accuracy_score(y_test,pred_lr),2),
        round(accuracy_score(y_test,pred_dt_test),2),
        round(accuracy_score(y_test,pred_rfc_test),2)
        ],
    'recall_score': [
        round(recall_score(y_test,pred_lr),2),
        round(recall_score(y_test,pred_dt_test),2),
        round(recall_score(y_test,pred_rfc_test),2)
    ],
    'roc_auc_score':[
        round(roc_auc_score(y_test,y_probs_lr),2),
        round(roc_auc_score(y_test,y_probs_dt),2),
        round(roc_auc_score(y_test,y_probs_rfc),2)
    ]
}

# Convering the Table into Pandas DataFrame
score_df = pd.DataFrame(score_table)
score_df

,Model,accuracy_score,recall_score,roc_auc_score
0,LogisticRegression,0.79,0.35,0.79
1,DecisionTree,0.78,0.22,0.70
2,RandomForest,0.78,0.37,0.71


`Logistic Regression` has the highest test ROC-AUC of the three models.

# Task 3.5 Tune, choose the model and the threshold, then test once

In [157]:
#  Finding the best parameters for Random Forest Classifier model 
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state= 42),
    {
        "n_estimators": [100, 200],
        "max_depth": [3,5,8]
    },
    cv = 5,
    scoring= "roc_auc"
)

rf_grid.fit(x_train_s,y_train)                                      # Fitting the model


print("Best Parameters: ", rf_grid.best_params_)
print("Best CV ROC-AUC: ", round(rf_grid.best_score_,2))

Best Parameters:  {'max_depth': 5, 'n_estimators': 200}
Best CV ROC-AUC:  0.77


In [158]:
# Storing the best score, and best estimator from Tuned Random Forest for future use.
best_rf_auc = rf_grid.best_score_
best_rf_model = rf_grid.best_estimator_

In [159]:
lr_pipe = Pipeline([                                                # Creating a Logistic Regression  pipeline, that Scales the data also
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter= 1000))
])

lr_auc = cross_val_score(                                           #  cross validation score for Linear Regression
    lr_pipe,
    x_train,
    y_train,
    cv=5,
    scoring='roc_auc'
).mean()

tree3_auc = cross_val_score(                                        # cross validation score for Decision Tree
    DecisionTreeClassifier(max_depth=3, random_state= 42),
    x_train,
    y_train,
    cv=5,
    scoring='roc_auc'
).mean()

rf_auc = cross_val_score(                                           # cross validation score for Random Forest
    RandomForestClassifier(
        n_estimators=100,
        random_state= 42
    ),
    x_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

print("Logistic Regression CV ROC-AUC:", lr_auc)
print("Depth-3 Tree CV ROC-AUC:", tree3_auc)
print("Random Forest CV ROC-AUC:", rf_auc)
print("Tuned Forest CV ROC-AUC:", best_rf_auc)


Logistic Regression CV ROC-AUC: 0.7744364096371686
Depth-3 Tree CV ROC-AUC: 0.6788259674544305
Random Forest CV ROC-AUC: 0.7403739462940601
Tuned Forest CV ROC-AUC: 0.7706518774078546


In [160]:
# calculating the best model
scores = {
    "Logistic Regression": lr_auc,
    "Depth-3 Tree": tree3_auc,
    "Random Forest": rf_auc,
    "Tuned Forest": best_rf_auc
}

winner_name = max(scores, key=scores.get)

print("Winning Model:", winner_name)
print("Best CV ROC-AUC:", scores[winner_name])

Winning Model: Logistic Regression
Best CV ROC-AUC: 0.7744364096371686


In [161]:
# Fine tuning Linear Regression model
oof = cross_val_predict(lr_pipe, x_train, y_train, cv=5, method="predict_proba")[:, 1]

final_model = lr_pipe

# Comparing the Threshold
for threshold in [0.50, 0.35]:
    pred = (oof >= threshold).astype(int)

    prec = precision_score(y_train, pred)
    recall = recall_score(y_train,pred)
    f1 = f1_score(y_train,pred)
    flagged = pred.sum()

    print('-'*50 )
    print(f"\nThreshold = {threshold}")
    print("Precision:", prec)
    print("Recall:", recall)
    print("F1:", f1)
    print("Number Flagged:", flagged)


--------------------------------------------------

Threshold = 0.5
Precision: 0.581081081081081
Recall: 0.25443786982248523
F1: 0.35390946502057613
Number Flagged: 74
--------------------------------------------------

Threshold = 0.35
Precision: 0.5131578947368421
Recall: 0.46153846153846156
F1: 0.48598130841121495
Number Flagged: 152


In [162]:
threshold_metrics = {}

for threshold in [0.50, 0.35]:

    pred = (oof >= threshold).astype(int)

    threshold_metrics[threshold] = f1_score(y_train,pred)

best_threshold = max(
    threshold_metrics,
    key=threshold_metrics.get
)

print("Selected Threshold:", best_threshold)                # Best Threshold 0.35

Selected Threshold: 0.35


We selected threshold __0.35__ because it achieved the higher `F1 score` and a high `Recall score`. Recall is especially important here because missing a likely no-show can leave an appointment slot empty, whereas contacting an extra patient costs only a few minutes of staff time.

`Linear Regression with threshold 0.35` is selected as the best Model for the data.

In [163]:
# Creating the Final Model
final_model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['age','distance_km','fee_inr',...,'booking_channel_Walk-In', 'visit_type_Consultation','visit_type_Follow-up']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [164]:
test_probs = final_model.predict_proba(x_test)[:,1]
test_pred = (test_probs >= best_threshold).astype(int)

In [165]:
# Calculating the Final scores for the model
precision = round(precision_score(y_test, test_pred),2)
recall = round(recall_score(y_test, test_pred),2)
roc_auc = round(roc_auc_score(y_test, test_probs),2)

flagged = test_pred.sum()

print("Precision:", precision)
print("Recall:", recall)
print("ROC-AUC:", roc_auc)
print("Number Flagged:", flagged)

Precision: 0.57
Recall: 0.57
ROC-AUC: 0.79
Number Flagged: 49


In [166]:
test_risk_scores = pd.DataFrame({
    "appointment_id":
        data.loc[x_test.index, "appointment_id"],
    "no_show":
        y_test,
    "risk_prob":
        test_probs,
    "flag":
        test_pred
})

# Storing the predicted result in test_risk_scores.csv file
test_risk_scores.to_csv(
    "test_risk_scores.csv",
    index=False
)

print(test_risk_scores.head())

    appointment_id  no_show  risk_prob  flag
213          A0217        0   0.022070     0
331          A0336        0   0.146586     0
501          A0508        0   0.520251     1
309          A0314        0   0.190753     0
88           A0089        0   0.165552     0


In practice, the threshold would be selected based on `operational capacity` for __*follow-up calls*__ and the *trade-off between increased recall* (catching more potential no-shows) *and the cost of spending staff time* contacting patients who would have attended anyway.